### Import Dependencies

In [2]:
import openai
import pandas as pd
from qdrant_client import QdrantClient, models
from qdrant_client.models import VectorParams, Distance, SparseVectorParams, Modifier,PayloadSchemaType, PointStruct, Document, Prefetch, FusionQuery

In [3]:
from dotenv import load_dotenv

load_dotenv('../../.env')

True

### Create new collection in Qdrant for hybrid search

In [4]:
qdrant_client = QdrantClient(url='http://localhost:6333')

In [23]:
qdrant_client.create_collection(
    collection_name='amazon-items-collection-01-hybrid-search',
    vectors_config={
        "text-embedding-3-small": VectorParams(size=1536, distance=Distance.COSINE),
    },
    sparse_vectors_config={
        "bm25": SparseVectorParams(modifier=Modifier.IDF)
    }
)

True

### Set index for hybrid search

In [24]:
qdrant_client.create_payload_index(
    collection_name='amazon-items-collection-01-hybrid-search',
    field_name='parent_asin',
    field_schema=PayloadSchemaType.KEYWORD
)

UpdateResult(operation_id=2, status=<UpdateStatus.COMPLETED: 'completed'>)

### Embedding functions

In [8]:
def get_embedding(text, model='text-embedding-3-small'):
    response = openai.embeddings.create(
        model=model,
        input=text
    )
    return response.data[0].embedding

In [17]:
def get_embeddings_batch(text_list, model='text-embedding-3-small', batch_size=100):
    if len(text_list) <= batch_size:
        response = openai.embeddings.create(input=text_list, model=model)
        return [embedding.embedding for embedding in response.data]
    
    all_embeddings = []
    counter = 1
    for i in range(0, len(text_list), batch_size):
        batch = text_list[i:i+batch_size]
        response = openai.embeddings.create(input=batch, model=model)
        all_embeddings.extend([embedding.embedding for embedding in response.data])
        print(f"Batch {counter * batch_size} completed of {len(text_list)}")
        counter += 1

    return all_embeddings

#### Read Sampled Data with Amazon items

In [9]:
df_items = pd.read_json('../../data/meta_Electronics_recent_2022_2023_has_main_category_ratings_100_sample_1000.jsonl', lines=True)

In [12]:
df_items.head()

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
0,AMAZON FASHION,"Bosttor Bluetooth Beanie Hat with Light, Headl...",4.5,2342,"[100% Acrylic, Elastic closure, Hand Wash Only...",[],26.99,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Beanie Hat with Bluetooth Headphon...,Bosttor,"[Electronics, Headphones, Earbuds & Accessorie...","{'Department': 'unisex-adult', 'Date First Ava...",B0BH41HYFZ,NaN,NaN,NaN
1,All Electronics,"Aceele USB and USB C to Ethernet Adapter, 3.3f...",4.2,148,[【USB or USB C to Ethernet Adapter】The Etherne...,[],14.99,[{'thumb': 'https://m.media-amazon.com/images/...,[],Aceele,"[Electronics, Computers & Accessories, Network...",{'Product Dimensions': '3.54 x 0.98 x 0.71 inc...,B0BKPB2YQ9,NaN,NaN,NaN
2,All Electronics,HDMI Switch 3 in 1 Out 4K UHD HDMI Switcher Sp...,4.2,3331,[📍3-Port HDMI Switch: This aluminum HDMI switc...,[],18.98,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Darren reviews VWRHAR 3-1 splitter...,VWRHar,"[Electronics, Home Audio, Home Audio Accessori...",{'Package Dimensions': '6.1 x 4.21 x 0.94 inch...,B09MM5QT3R,NaN,NaN,NaN
3,All Electronics,"Smart Watch,Ip67 Waterproof Bluetooth Smartwat...",3.4,125,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Burxoe,[],{'Package Dimensions': '3.15 x 3.07 x 2.52 inc...,B09Q5TNDHY,NaN,NaN,NaN
4,Cell Phones & Accessories,"(3 Pack) Cute Airpod Case for Airpods 2&1,3D D...",4.7,335,[【Compatible Airpods 1/2 】This case designed f...,[1],11.99,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'airpod cases', 'url': 'https://www...",UGUHY,"[Electronics, Headphones, Earbuds & Accessorie...",{'Package Dimensions': '7.44 x 4.61 x 1.57 inc...,B0BZJKX7MS,NaN,NaN,NaN


### Preprocess title and features

In [10]:
def preprocess_description(row):
    return f"{row['title']} {' '.join(row['features'])}"

In [11]:
def extract_first_large_image(row):
    return row['images'][0].get('large','')

In [13]:
df_items['processed_description'] = df_items.apply(preprocess_description, axis=1)
df_items['image'] = df_items.apply(extract_first_large_image, axis=1)

### Add data to Qdrant collection

In [14]:
relevant_columns = ['processed_description', 'image', 'rating_number', 'price', 'average_rating','parent_asin']
df_data_to_embed = df_items[relevant_columns]

In [15]:
data_to_embed = df_data_to_embed.to_dict(orient='records')

In [16]:
text_to_embed = [item['processed_description'] for item in data_to_embed]

In [18]:
embeddings = get_embeddings_batch(text_to_embed)

Batch 100 completed of 1000
Batch 200 completed of 1000
Batch 300 completed of 1000
Batch 400 completed of 1000
Batch 500 completed of 1000
Batch 600 completed of 1000
Batch 700 completed of 1000
Batch 800 completed of 1000
Batch 900 completed of 1000
Batch 1000 completed of 1000


In [19]:
len(embeddings)

1000

In [20]:
point_structs = []
i = 1
for embedding, data in zip(embeddings, data_to_embed):
    point_structs.append(
        PointStruct(
            id=i,
            vector={
                "text-embedding-3-small": embedding,
                "bm25": Document(
                    text=data['processed_description'],
                    model="qdrant/bm25",
                )
            },
            payload=data
        )
    )
    i += 1

In [25]:
qdrant_client.upsert(
    collection_name='amazon-items-collection-01-hybrid-search',
    points=point_structs[:500],
    wait=True
)

UpdateResult(operation_id=3, status=<UpdateStatus.COMPLETED: 'completed'>)

In [26]:
qdrant_client.upsert(
    collection_name='amazon-items-collection-01-hybrid-search',
    points=point_structs[500:],
    wait=True
)

UpdateResult(operation_id=4, status=<UpdateStatus.COMPLETED: 'completed'>)

### Hybrid retrieval

In [27]:
def retrieve_data(query, collection_name='amazon-items-collection-01-hybrid-search', k=5):
    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name=collection_name,
        prefetch=[
            Prefetch(
                query=query_embedding,
                using="text-embedding-3-small",
                limit=20
            ),
            Prefetch(
                query=Document(
                    text=query,
                    model="qdrant/bm25",
                ),
                using="bm25",
                limit=20
            )
        ],
        query=FusionQuery(fusion="rrf"),
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context_scores = []
    retrieved_context_texts = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload['parent_asin'])
        retrieved_context_scores.append(result.score)
        retrieved_context_texts.append(result.payload['processed_description'])
        retrieved_context_ratings.append(result.payload['average_rating'])

    return {
        'retrieved_context_ids': retrieved_context_ids,
        'retrieved_context_scores': retrieved_context_scores,
        'retrieved_context_texts': retrieved_context_texts,
        'retrieved_context_ratings': retrieved_context_ratings
    }


In [28]:
results = retrieve_data("Can I a get a tablet?", k=20)

In [32]:
results

{'retrieved_context_ids': ['B0BN58Z4YX',
  'B0C7DCS2KW',
  'B09RHF4L45',
  'B0B157WDDJ',
  'B0BGHBL86V',
  'B0BHVH4D37',
  'B0B159KDFP',
  'B0BSD3QK7M',
  'B0B58CPFWY',
  'B0B6ZZH83Y',
  'B0BCFYCXRH',
  'B09WR36NK8',
  'B0BT83RRJ2',
  'B09XQMRYBJ',
  'B07Y36DDYM',
  'B09SZNLXJQ',
  'B0B2KJKT86',
  'B0BWRZ1HRD',
  'B0B3R74TZC',
  'B0BZCM9CBR'],
 'retrieved_context_scores': [0.5909091,
  0.5909091,
  0.43333334,
  0.3611111,
  0.33333334,
  0.33333334,
  0.325,
  0.25396827,
  0.25,
  0.2,
  0.18333334,
  0.14285715,
  0.125,
  0.083333336,
  0.07692308,
  0.07692308,
  0.071428575,
  0.071428575,
  0.06666667,
  0.06666667],
 'retrieved_context_texts': ['ASWINN Tablet Tripod Stand, Gooseneck 65" Height Adjustable Tablet Stand Floor with 360° Rotating Tripod Mount Suitable for iPhone,Tablet,Kindle and All 4.5-12.9 Inch Tablet and Phone (Black) 【Free Your Hands】: With this gooseneck arm tablet tripod stand, you can find a more comfortable way to use tablet & Phone in bed or sofa in winter

### Hybrid search with weighthed RRF

In [33]:
def retrieve_data(query, collection_name='amazon-items-collection-01-hybrid-search', k=5):
    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name=collection_name,
        prefetch=[
            Prefetch(
                query=query_embedding,
                using="text-embedding-3-small",
                limit=20
            ),
            Prefetch(
                query=Document(
                    text=query,
                    model="qdrant/bm25",
                ),
                using="bm25",
                limit=20
            )
        ],
        query=models.RrfQuery(rrf=models.Rrf(weights=[3,1])),
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context_scores = []
    retrieved_context_texts = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload['parent_asin'])
        retrieved_context_scores.append(result.score)
        retrieved_context_texts.append(result.payload['processed_description'])
        retrieved_context_ratings.append(result.payload['average_rating'])

    return {
        'retrieved_context_ids': retrieved_context_ids,
        'retrieved_context_scores': retrieved_context_scores,
        'retrieved_context_texts': retrieved_context_texts,
        'retrieved_context_ratings': retrieved_context_ratings
    }


In [34]:
results = retrieve_data("Can I a get a tablet?", k=20)

In [35]:
results

{'retrieved_context_ids': ['B0C7DCS2KW',
  'B0BN58Z4YX',
  'B0B157WDDJ',
  'B0BGHBL86V',
  'B09RHF4L45',
  'B0B159KDFP',
  'B0BHVH4D37',
  'B0BSD3QK7M',
  'B09WR36NK8',
  'B0BCFYCXRH',
  'B0BT83RRJ2',
  'B0B58CPFWY',
  'B0B6ZZH83Y',
  'B07Y36DDYM',
  'B0BWRZ1HRD',
  'B0B3R74TZC',
  'B09PYV6B4B',
  'B0B96TTN6T',
  'B08SC3C9MC',
  'B09RN3KN5C'],
 'retrieved_context_scores': [0.8409091,
  0.7307693,
  0.6111111,
  0.59999996,
  0.5833334,
  0.5535714,
  0.5416667,
  0.4155844,
  0.33333334,
  0.3142857,
  0.30000004,
  0.25,
  0.2,
  0.2,
  0.1875,
  0.1764706,
  0.16666667,
  0.15789473,
  0.15,
  0.14285715],
 'retrieved_context_texts': ['KYASTER Kids Tablet, 7 inch 5G WiFi 6 Android 12 Tablet for Kids, Full HD 1920x1200 IPS Screen, 2GB RAM 32GB ROM,Parental Controls Game Education Apps,EVA Kids-Proof Case with Stylus 💗【Full HD 1920*1200 IPS Touch Screen】 7 inch IPS touch screen with native 1080p resolution display,178° wide view angle,Clear bright screen provides outstanding visual exp